# Advanced Session: Evaluating Energy Above Hull
AI for Materials Science — Hands-on session 1

Section C of the in-class notebook built a phase diagram and printed `energy_above_hull` for every
entry in the table. This notebook is about that one number.

Energy above hull is the most used stability descriptor in computational materials screening. It
decides which candidates are carried forward and which are dropped, so it is worth knowing exactly
what it measures, how to compute it yourself, and where it stops being informative.

## Today's plan

### 0 · Setup
Install the libraries and enter your MP API key.

### A · What the number measures
Read the three energies the Materials Project reports for one composition, and work out which of
them can be compared across materials.

### B · Ranking structures at one composition
The LiFePO4 composition holds dozens of calculated structures. Sort them by hull distance and count
how many sit within thermal energy of the ground state.

### C · Computing the hull yourself
Build the Li-Fe-P-O hull from queried entries and reproduce the database's `energy_above_hull`.

### D · Reading a hull distance as a decomposition reaction
A phase above the hull decomposes into something specific. Turn the returned weights into a
balanced reaction.

### E · The hull depends on the entry set
Remove one competitor and the same material changes its hull distance. This is the part that
matters when you screen.

---

Run the cells one at a time from the top.
Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not
defined" error.
Lines starting with `##` inside the code are comments written for you; Python does not run them.

Unlike the preclass and in-class notebooks, this one reads no files from the course repository.
Every number comes from a live query, so there is no `git clone` step and **an MP API key is
required**.

Short practice cells are placed at the end of each section. There is nothing to submit.

## 0. Setup

Two things to get in place: the libraries and your MP API key.

### 0-1. Install the libraries
`pymatgen` builds the convex hull and reads compositions, and `mp_api` queries the Materials
Project. NumPy, pandas and matplotlib come along with them.

In [ ]:
## A leading ! runs a terminal command instead of Python. -q keeps the install output quiet.
!pip install -q pymatgen mp_api

### 0-2. Import the libraries
Installing and importing are two different steps. Installing puts files on the machine; `import`
brings a tool into this notebook.

In [ ]:
## Standard Python tools for environment variables, file paths and hidden key entry.
import os
from pathlib import Path
from getpass import getpass

## np for numeric work, pd for tables, plt for figures.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Composition parses a chemical formula; PhaseDiagram builds the convex hull.
from pymatgen.core import Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram

## The Materials Project query client.
from mp_api.client import MPRester

### 0-3. Set the output folder
Every table and figure from today is written here.

In [ ]:
## parents=True creates the outputs folder too; exist_ok=True makes a second run harmless.
OUTPUT = Path("outputs/03_advanced")
OUTPUT.mkdir(parents=True, exist_ok=True)

print("outputs:", OUTPUT.resolve())

### 0-4. Enter your MP API key
Every section of this notebook queries the Materials Project directly, and there is no saved copy to
fall back on, so the key is required. You can get one for free from the
[MP account page](https://next-gen.materialsproject.org/api).

Treat the key like a password. Written into a code cell or a submitted file, it is exposed.
`getpass` takes the input without echoing it to the screen. If you would rather not type it every
time, set `MP_API_KEY` in the environment before starting the notebook and this cell will pick it up.

The cell prints the database version it connected to. Write it down: entries are added continuously,
so hull distances shift slightly between versions, and the version is part of any number you report.

In [ ]:
## Use the key from an environment variable if it is set; otherwise ask for it directly.
API_KEY = os.getenv("MP_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Materials Project API key: ").strip()

## This notebook has no offline path, so stop here rather than failing halfway through.
if not API_KEY:
    raise ValueError("An API key is required. Run this cell again and enter your key.")

## One connection up front confirms the key works before any real query runs.
with MPRester(API_KEY) as mpr:
    MP_DB_VERSION = mpr.db_version

print("Connected. MP database version:", MP_DB_VERSION)

## A. What energy above hull measures

The Materials Project reports several energies for every material and they are not interchangeable.
Section A is about telling them apart.

We use the LiFePO4 composition throughout the notebook. It is the cathode material from the preclass
notebook, and the database holds many calculated structures at that one composition, which is
exactly what we need.

### A-1. Query one composition
`summary.search` returns one document per material. `formula` matches on the reduced formula, so a
single call collects every structure at this composition. `fields` limits what comes back; without it
the response is far larger than we need.

In [ ]:
## The fields we need to put the three energies side by side.
SUMMARY_FIELDS = ["material_id", "formula_pretty", "symmetry",
                  "energy_per_atom", "formation_energy_per_atom",
                  "energy_above_hull", "is_stable"]

with MPRester(API_KEY) as mpr:
    lfp_docs = mpr.materials.summary.search(formula="LiFePO4", fields=SUMMARY_FIELDS)

print("Structures at the LiFePO4 composition:", len(lfp_docs))

### A-2. Three energies, one table
Every row below is a **different calculated structure at the same composition**.

In [ ]:
## One dictionary per document becomes one row of the table.
lfp_rows = []
for doc in lfp_docs:
    lfp_rows.append({
        "material_id": doc.material_id,
        ## symmetry.symbol is the space group. str() keeps it as plain text in the table.
        "spacegroup": str(doc.symmetry.symbol),
        "energy_eV_atom": doc.energy_per_atom,
        "formation_energy_eV_atom": doc.formation_energy_per_atom,
        "e_above_hull_eV_atom": doc.energy_above_hull,
        "is_stable": doc.is_stable,
    })

## Sorting by hull distance puts the ground state in the first row.
lfp_summary = (pd.DataFrame(lfp_rows)
               .sort_values("e_above_hull_eV_atom")
               .reset_index(drop=True))

lfp_summary.head(10)

Read the three energy columns from left to right.

- `energy_eV_atom` — the total DFT energy of that calculation divided by the number of atoms. It
  carries the arbitrary reference of the pseudopotentials, so on its own the value means nothing.
  **Never compare this number between two different compositions.**
- `formation_energy_eV_atom` — the same energy measured from the elemental references. Negative means
  the compound is downhill from the elements. This one *is* comparable across compositions, but it
  does not tell you whether some other compound is further downhill still.
- `e_above_hull_eV_atom` — the distance to the convex hull of the whole chemical system. **Zero means
  nothing in the database beats it.** A positive value is the energy per atom released by letting the
  phase fall apart into whatever does beat it.

The third column is the one used for screening, because it is the only one that answers the question
"is there something better at this composition?".

### A-3. Zero is the interesting value
The ground state should be the row with a hull distance of zero, and it should be the only row with
`is_stable` set. Check both.

In [ ]:
## idxmin returns the row label of the smallest value, so this picks the ground state.
ground_state_row = lfp_summary.loc[lfp_summary["e_above_hull_eV_atom"].idxmin()]

print("Ground state:", ground_state_row["material_id"],
      "| spacegroup", ground_state_row["spacegroup"])
print("Its hull distance:", ground_state_row["e_above_hull_eV_atom"], "eV/atom")
print()
## True counts as 1 when summed, so this counts the flagged rows.
print("Rows flagged is_stable :", int(lfp_summary["is_stable"].sum()), "of", len(lfp_summary))
## Floating point rarely gives exactly 0, so compare against a small tolerance instead.
print("Rows below 1e-6 eV/atom:", int((lfp_summary["e_above_hull_eV_atom"] < 1e-6).sum()))

### A-4. What "the hull" is
Plot formation energy per atom against composition for every calculated phase in a chemical system.
The **convex hull** is the lower envelope of that cloud of points: the phases and phase mixtures that
no combination of other phases can undercut.

For an entry at composition $c$,

$$E_{\rm hull}(c) = E_{f}(c) - E_{f}^{\rm hull}(c)$$

where $E_f^{\rm hull}(c)$ is the height of the envelope at that composition. An entry sitting on the
envelope gives zero. We check this identity numerically in C-4.

Two consequences worth holding on to.

- Hull distance is a **vertical** distance at fixed composition. It is not a distance to the nearest
  stable compound.
- It is defined **relative to a set of competing phases**. Change the set and the number changes,
  which is what section E is about.

### A-5. Practice 1
`energy_eV_atom` and `formation_energy_eV_atom` differ by the elemental reference term.

1. Take the first two rows of `lfp_summary`. Confirm that the difference between their
   `energy_eV_atom` values equals the difference between their `formation_energy_eV_atom` values.
2. Explain why that has to hold for two rows at the same composition.
3. Would it still hold for two rows at different compositions? Say why in one sentence.

In [ ]:
## Write your code here.

## B. Ranking structures at one composition

At a fixed composition, hull distance becomes a straight ranking of structures: the ground state at
zero, and every other polymorph measured up from it.

This is how you decide which of many calculated structures is the one to talk about, and how you
find out whether that choice was even close.

### B-1. The whole ranking
Hull distances at this scale are easier to read in meV/atom than in eV/atom, so we add a converted
column.

In [ ]:
## copy() makes a table separate from lfp_summary, so adding a column does not touch the original.
lfp_ranking = lfp_summary[["material_id", "spacegroup", "e_above_hull_eV_atom"]].copy()
## 1 eV = 1000 meV. The whole column is converted in one operation.
lfp_ranking["e_above_hull_meV_atom"] = lfp_ranking["e_above_hull_eV_atom"] * 1000

print("Structures in the ranking:", len(lfp_ranking))
print("Ground state             :", lfp_ranking.iloc[0]["material_id"])
print("Highest in this set      :",
      round(lfp_ranking["e_above_hull_meV_atom"].max(), 1), "meV/atom above the hull")

lfp_ranking.head(15)

### B-2. How close is close?
Thermal energy at room temperature is $k_{\rm B}T \approx 26$ meV. A polymorph a few tens of
meV/atom above the ground state is **near-degenerate**: the calculation ranks it below, but the gap
is comparable to the error of the method and well inside what temperature, strain or a different
synthesis route can reorder.

Counting how many structures fall inside such a window tells you how much confidence the ranking
deserves. A ground state 300 meV/atom clear of everything else is a firm result; a ground state with
ten structures within 20 meV/atom is a coin toss.

In [ ]:
## Boltzmann's constant in eV/K. Multiplying by 300 K and by 1000 gives meV.
BOLTZMANN_eV_PER_K = 8.617333262e-5
kT_300K_meV = BOLTZMANN_eV_PER_K * 300 * 1000
print(f"kT at 300 K: {kT_300K_meV:.1f} meV")
print()

## Count how many structures fall inside each window. 0.001 meV stands in for "on the hull".
window_rows = []
for cut_meV in [0.001, 10, 25, 50, 100, 200]:
    inside = int((lfp_ranking["e_above_hull_meV_atom"] <= cut_meV).sum())
    window_rows.append({"window_meV_atom": cut_meV,
                        "structures_inside": inside,
                        "fraction_of_set": inside / len(lfp_ranking)})

pd.DataFrame(window_rows)

### B-3. The shape of the ranking
Plotting hull distance against rank shows the structure of the set without having to pick histogram
bins. The dashed line marks $k_{\rm B}T$ at 300 K, so anything below it is inside thermal energy of
the ground state.

Two features to look for. How many points sit below the line, and where the first large jump comes.
A dense cluster near zero means the ranking is not resolved by the calculation; an early jump means
the ground state is a firm result.

In [ ]:
## fig is the whole figure, ax is the plotting area.
fig, ax = plt.subplots(figsize=(7.2, 4.2))

## lfp_ranking is already sorted, so the row position is the rank.
ranks = np.arange(1, len(lfp_ranking) + 1)
ax.plot(ranks, lfp_ranking["e_above_hull_meV_atom"],
        marker="o", markersize=4, linewidth=1.2, color="#4a6fa5")
## axhline draws a horizontal reference line across the plot.
ax.axhline(kT_300K_meV, linestyle="--", color="#b3261e", linewidth=1.4,
           label=f"kT at 300 K = {kT_300K_meV:.0f} meV")

## The highest structure is far above the cluster near zero. symlog is a log scale that still
## accepts the value 0, so both ends of the range stay readable in one panel.
ax.set_yscale("symlog", linthresh=1)
## Cutting the axis at zero keeps the panel from reserving space for negative values.
ax.set_ylim(0, lfp_ranking["e_above_hull_meV_atom"].max() * 1.3)
ax.set_xlabel("Rank at the LiFePO4 composition")
ax.set_ylabel("Energy above hull (meV/atom)")
ax.set_title(f"LiFePO4 structures in MP {MP_DB_VERSION}")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT / "lfp_hull_distance_ranking.png", dpi=180)
plt.show()

### B-4. Practice 2
Repeat B-1 to B-3 for a different composition. `formula="Fe2O3"` and `formula="FePO4"` both have
plenty of structures in the database, and you can reuse `SUMMARY_FIELDS` unchanged.

1. How many structures does the database hold at your composition?
2. How many sit within 25 meV/atom of the ground state?
3. Is the ground state well separated from the rest, or is the ranking close enough that you would
   not trust it?

In [ ]:
## Write your code here.

## C. Computing the hull yourself

So far we read `energy_above_hull` straight out of the database. Now we build the hull from the
underlying entries and reproduce that number.

This matters for two reasons. Any composition you calculate yourself has no database entry, so to
place it you have to build a hull. And reproducing a value you can also look up is the cheapest
possible check that your own hull is set up correctly.

### C-1. Fetching every subsystem
A hull over Li-Fe-P-O needs more than the quaternary compounds. It needs the **elemental references**
(Li, Fe, P, O2) and every **competing binary and ternary** — Li2O, FePO4, Li3PO4 and the rest.
Without them there is nothing for the envelope to be built against. `get_entries_in_chemsys` returns
all of it in one call.

Two arguments are worth understanding.

- `compatible_only=True` applies the energy corrections MP fits for oxides and for GGA+U transition
  metals. The hull assumes corrected energies, so leaving this off changes the answer.
- `additional_criteria={"thermo_types": ["GGA_GGA+U"]}` pins the calculation family. Left out, the
  query returns MP's **mixed** set, where r2SCAN energies replace the GGA+U ones for materials that
  have them. Both are legitimate hulls, but they are different hulls. We pin GGA/GGA+U so our numbers
  can be compared with the summary values from section A.

The query takes a few seconds.

In [ ]:
PD_ELEMENTS = ["Li", "Fe", "P", "O"]

with MPRester(API_KEY) as mpr:
    pd_entries = mpr.get_entries_in_chemsys(
        PD_ELEMENTS, compatible_only=True,
        additional_criteria={"thermo_types": ["GGA_GGA+U"]})

print("Entries returned:", len(pd_entries))

## A set removes duplicates, so this lists each elemental reference once. All four must appear.
element_references = sorted({e.composition.reduced_formula
                             for e in pd_entries if e.composition.is_element})
print("Elemental references:", element_references)

## Count entries by how many elements they contain, to see the subsystems that came back.
element_counts = pd.Series([len(e.composition.elements) for e in pd_entries]).value_counts()
print("Entries by number of elements:", element_counts.sort_index().to_dict())

### C-2. What happens if you leave the subsystems out
It is worth seeing this failure rather than being told about it. Keeping only the four-element
compounds throws away the elemental references, and `PhaseDiagram` refuses to build.

In [ ]:
## Keep only entries that contain all four elements.
quaternary_only = [e for e in pd_entries if len(e.composition.elements) == 4]
print("Quaternary-only entries:", len(quaternary_only))

## try/except prints the error instead of stopping the notebook.
try:
    PhaseDiagram(quaternary_only)
    print("This line should not be reached.")
except ValueError as error:
    print("PhaseDiagram refused:", error)

There are hundreds of entries in that list and the hull still cannot be built. Quantity does not
substitute for the terminal references.

### C-3. Building the hull and reproducing the database
`PhaseDiagram` takes the full entry list and computes the envelope. `get_e_above_hull` then places
any entry against it.

One detail about identifiers: entries from `get_entries_in_chemsys` carry a calculation suffix, so
`mp-19017` comes back as `mp-19017-GGA+U`. Strip the suffix before matching against the
`material_id` values from section A.

In [ ]:
phase_diagram = PhaseDiagram(pd_entries)
print("Stable phases on this hull:", len(phase_diagram.stable_entries))
print()

## split("-GGA")[0] turns mp-19017-GGA+U back into mp-19017.
own_rows = []
for entry in pd_entries:
    if entry.composition.reduced_formula == "LiFePO4":
        own_rows.append({
            "material_id": str(entry.entry_id).split("-GGA")[0],
            "own_e_above_hull_eV_atom": phase_diagram.get_e_above_hull(entry),
        })
own_hull = pd.DataFrame(own_rows)

## merge lines the two tables up on material_id, the column they share.
comparison = lfp_summary[["material_id", "e_above_hull_eV_atom"]].merge(
    own_hull, on="material_id", how="inner")
comparison["difference_meV_atom"] = (
    (comparison["own_e_above_hull_eV_atom"] - comparison["e_above_hull_eV_atom"]) * 1000)

print("Rows matched         :", len(comparison), "of", len(lfp_summary))
print("Largest disagreement :",
      round(comparison["difference_meV_atom"].abs().max(), 6), "meV/atom")

comparison.head(10)

A disagreement below about 1 meV/atom means you built the same hull the database used.

A disagreement of tens of meV/atom almost always means a different entry set: a missing subsystem, a
different `thermo_types`, or `compatible_only=False`. It is worth running this check once whenever you
set up a new system, because a hull that is quietly wrong still produces a full table of plausible
numbers.

### C-4. The identity from A-4, checked
`get_hull_energy_per_atom` returns the height of the envelope at a given composition. Subtracting it
from an entry's own energy per atom has to give the hull distance.

In [ ]:
## The second row of the ranking is the runner-up structure at this composition.
runner_up_id = lfp_ranking.iloc[1]["material_id"]
## next() takes the first match from the generator, so this finds that entry in the list.
runner_up = next(e for e in pd_entries
                 if str(e.entry_id).split("-GGA")[0] == runner_up_id)

hull_height = phase_diagram.get_hull_energy_per_atom(runner_up.composition)
by_hand = runner_up.energy_per_atom - hull_height
from_pymatgen = phase_diagram.get_e_above_hull(runner_up)

print("Entry                       :", runner_up_id)
print("Its energy per atom         :", round(runner_up.energy_per_atom, 6), "eV/atom")
print("Hull height at that composition:", round(hull_height, 6), "eV/atom")
print("Difference                  :", round(by_hand, 6), "eV/atom")
print("get_e_above_hull            :", round(from_pymatgen, 6), "eV/atom")
## np.isclose compares two floats while allowing for rounding error.
print("Identity holds              :", bool(np.isclose(by_hand, from_pymatgen)))

### C-5. Practice 3
`phase_diagram` covers the whole Li-Fe-P-O system, so it already contains every FePO4 entry. No second
`get_entries_in_chemsys` call is needed.

1. Query the summary documents for `formula="FePO4"` the way A-1 did for LiFePO4.
2. Build the C-3 comparison table for FePO4 and report the largest disagreement.
3. In one sentence, say why you could reuse `phase_diagram` without querying the entries again.

In [ ]:
## Write your code here.

## D. Reading a hull distance as a decomposition reaction

A positive hull distance is not just a score. It says the phase is unstable **against a specific set
of products**, and pymatgen will tell you which ones.

`get_decomp_and_e_above_hull` returns both at once: the products with their weights, and the distance.
The products are the part that turns a number into chemistry — they tell you what you would actually
find in the crucible.

### D-1. Picking a target above the hull
We use the Li4Fe2(PO4)3 composition, which sits between LiFePO4 and Li3PO4 in this system. Rather
than hard-coding a material ID that could change between database versions, we take the
lowest-energy structure at that composition.

In [ ]:
TARGET_FORMULA = "Li4Fe2(PO4)3"

## Every structure the C-1 query returned at this composition.
target_candidates = [e for e in pd_entries
                     if e.composition.reduced_formula == TARGET_FORMULA]
if not target_candidates:
    raise LookupError(f"No entries at the {TARGET_FORMULA} composition in this query.")

## min with key picks the entry with the smallest hull distance.
target = min(target_candidates, key=phase_diagram.get_e_above_hull)
decomposition, target_e_hull = phase_diagram.get_decomp_and_e_above_hull(target)

print("Structures at this composition:", len(target_candidates))
print("Target entry     :", target.entry_id)
print("Hull distance    :", round(float(target_e_hull), 6), "eV/atom")
print("Number of products:", len(decomposition))

### D-2. Weights are atomic fractions, not reaction coefficients
The weights that come back say what fraction of the **atoms** ends up in each product. To write a
balanced reaction you have to convert them to coefficients per formula unit:

$$\text{coefficient} = \text{weight} \times \frac{N_{\rm target}}{N_{\rm product}}$$

where $N$ is the number of atoms in the reduced formula. Li4Fe2(PO4)3 reduces to 21 atoms, so a
product with 4 atoms per formula unit and a weight of 1/3 gets a coefficient of $1/3 \times 21/4$.

In [ ]:
## num_atoms on the reduced formula, not on the calculation cell, is what the conversion needs.
target_atom_count = Composition(TARGET_FORMULA).num_atoms

decomposition_rows = []
## .items() walks a dictionary as (key, value) pairs. Here the key is the product entry.
for product, weight in decomposition.items():
    product_formula = product.composition.reduced_formula
    product_atom_count = Composition(product_formula).num_atoms
    decomposition_rows.append({
        "product_formula": product_formula,
        "product_entry_id": str(product.entry_id).split("-GGA")[0],
        "atoms_per_formula_unit": product_atom_count,
        "atomic_fraction_weight": float(weight),
        "coefficient_per_formula_unit": (float(weight) * target_atom_count
                                         / product_atom_count),
    })

decomposition_table = (pd.DataFrame(decomposition_rows)
                       .sort_values("product_formula")
                       .reset_index(drop=True))

print(f"{TARGET_FORMULA} has {target_atom_count:.0f} atoms per formula unit")
print("Atomic fractions sum to:", round(decomposition_table["atomic_fraction_weight"].sum(), 6))

decomposition_table

The atomic fractions sum to 1 because every atom has to go somewhere. The coefficients do not sum to
anything in particular; they are the numbers you put in front of each product.

### D-3. Writing the reaction out and checking it
A decomposition reaction is only correct if every element balances. The cell below assembles the
reaction string and then checks the atom counts on both sides.

In [ ]:
## Build the right-hand side as text, leaving out a coefficient of exactly 1.
product_terms = []
for row in decomposition_table.itertuples():
    coefficient = row.coefficient_per_formula_unit
    prefix = "" if np.isclose(coefficient, 1.0) else f"{coefficient:.4g} "
    product_terms.append(f"{prefix}{row.product_formula}")

print(f"{TARGET_FORMULA} -> " + " + ".join(product_terms))
print()

## Sum coefficient x atom count for each element on the product side.
product_atoms = {}
for row in decomposition_table.itertuples():
    for element, amount in Composition(row.product_formula).get_el_amt_dict().items():
        ## get(element, 0.0) starts a newly seen element at zero instead of raising an error.
        product_atoms[element] = (product_atoms.get(element, 0.0)
                                  + row.coefficient_per_formula_unit * amount)

target_atoms = Composition(TARGET_FORMULA).get_el_amt_dict()
balance = pd.DataFrame([
    {"element": element,
     "left_side": target_atoms.get(element, 0.0),
     "right_side": product_atoms.get(element, 0.0)}
    for element in sorted(set(target_atoms) | set(product_atoms))
])
balance["balanced"] = np.isclose(balance["left_side"], balance["right_side"])

print("Every element balanced:", bool(balance["balanced"].all()))
balance

### D-4. Composition fixes the products
Every structure at the LiFePO4 composition that sits above the hull decomposes into the same thing:
the ground state at that composition. The distance differs from structure to structure, but the
products cannot, because the composition is identical.

That is the difference between the two cases. A polymorph competes only with other structures at its
own composition, while Li4Fe2(PO4)3 in D-1 competes with a **mixture** of three other compositions.

In [ ]:
lfp_decomposition_rows = []
for entry in pd_entries:
    ## continue skips to the next loop iteration, so only LiFePO4 entries are processed.
    if entry.composition.reduced_formula != "LiFePO4":
        continue
    products, e_hull = phase_diagram.get_decomp_and_e_above_hull(entry)
    lfp_decomposition_rows.append({
        "material_id": str(entry.entry_id).split("-GGA")[0],
        "e_above_hull_eV_atom": float(e_hull),
        "n_products": len(products),
        "products": " + ".join(sorted(p.composition.reduced_formula for p in products)),
    })

lfp_decomposition = (pd.DataFrame(lfp_decomposition_rows)
                     .sort_values("e_above_hull_eV_atom")
                     .reset_index(drop=True))

## nunique counts how many different values a column holds.
print("Structures checked   :", len(lfp_decomposition))
print("Distinct product sets:", lfp_decomposition["products"].nunique())
print()
print(lfp_decomposition["products"].value_counts().to_string())

lfp_decomposition.head(5)

### D-5. Practice 4
Pick a different composition above the hull and write out its decomposition reaction. `Fe2PO5` and
`Li2FeP2O7` both work, and both are already in `pd_entries`. Change `TARGET_FORMULA` and rerun D-1
to D-3.

1. How many products does your composition decompose into?
2. Do the coefficients balance for every element?
3. Li-Fe-P-O has four components. From the geometry of a hull in a four-component system, what is the
   largest number of products you would expect any single composition to decompose into?

In [ ]:
## Write your code here.

## E. The hull depends on the entry set

This is the section that changes how you read a screening result.

`energy_above_hull` is not a property of a material. It is a property of a material **measured against
a list of competitors**. Change the list and the number changes, with no new calculation of the
material itself.

### E-1. Remove the ground state
Take the runner-up at the LiFePO4 composition from C-4, drop the ground state from the entry list,
and rebuild the hull.

In [ ]:
## The ground state at this composition is the LiFePO4 entry closest to the hull.
ground_state_entry = min((e for e in pd_entries
                          if e.composition.reduced_formula == "LiFePO4"),
                         key=phase_diagram.get_e_above_hull)

## "is not" compares identity, so this drops that one object and keeps every other entry.
without_ground_state = [e for e in pd_entries if e is not ground_state_entry]
hull_without_ground_state = PhaseDiagram(without_ground_state)

print("Removed:", str(ground_state_entry.entry_id).split("-GGA")[0])
print("Entries used :", len(pd_entries), "->", len(without_ground_state))
print("Stable phases:", len(phase_diagram.stable_entries), "->",
      len(hull_without_ground_state.stable_entries))
print()
print("Runner-up", runner_up_id, "hull distance")
print("  with the ground state present:",
      round(phase_diagram.get_e_above_hull(runner_up), 6), "eV/atom")
print("  with it removed              :",
      round(hull_without_ground_state.get_e_above_hull(runner_up), 6), "eV/atom")

The runner-up became stable, and nothing about it was recalculated.

The count of stable phases did not move, because the runner-up stepped straight into the vacancy the
ground state left. What changed is which structure the hull reports as the LiFePO4 ground state.

A hull distance of zero means "nothing in this entry set beats it", and one entry left the set. This
is exactly the position you are in when you screen a new composition: the ground state at that
composition may simply not have been calculated yet.

### E-2. Remove a competitor instead
The section D target decomposes into three phases. Drop **every** structure at one of those product
compositions and rebuild. Dropping a single entry would not be enough, because another polymorph at
the same composition would step into its place.

In [ ]:
COMPETITOR_FORMULA = "Li3Fe2(PO4)3"

## != keeps everything whose reduced formula is not the competitor.
without_competitor = [e for e in pd_entries
                      if e.composition.reduced_formula != COMPETITOR_FORMULA]
hull_without_competitor = PhaseDiagram(without_competitor)

new_products, new_e_hull = hull_without_competitor.get_decomp_and_e_above_hull(target)

print("Entries removed:", len(pd_entries) - len(without_competitor))
print()
print(f"{TARGET_FORMULA} hull distance")
print("  full entry set :", round(float(target_e_hull), 6), "eV/atom")
print("  competitor gone:", round(float(new_e_hull), 6), "eV/atom")
print()
print("Products, full entry set :",
      " + ".join(sorted(p.composition.reduced_formula for p in decomposition)))
print("Products, competitor gone:",
      " + ".join(sorted(p.composition.reduced_formula for p in new_products)))

The target moved closer to the hull and its decomposition route changed, again without recalculating
the target.

The direction of the effect is always the same. **Removing competitors can only lower a hull distance,
and adding them can only raise it.** A hull built from an incomplete entry set therefore reports
candidates as *more* stable than they are, which is the failure mode to watch for in a chemical system
the database has not covered densely.

### E-3. What this means when you screen
Four practical rules follow from sections A to E.

- **Never compare hull distances across differently built hulls.** Same chemical system, same
  `thermo_types`, same correction scheme, or the numbers are not on the same axis.
- **Report the database version.** Entries are added continuously, so a hull distance is a statement
  about a snapshot, not a constant of nature.
- **Read a small hull distance as "not resolved" rather than "unstable".** A few tens of meV/atom is
  inside the error of the method, as B-2 showed.
- **Hull distance ranks thermodynamic driving force, not synthesizability.** It says nothing about the
  barrier to decomposition, and materials well above the hull are made and used routinely.

One more limit covers all of it: everything on this hull is a 0 K electronic-structure result, with
the solid $pV$ term treated as negligible. Temperature, pressure and reaction kinetics are not in it.
A phase reported as unstable at 0 K can be the equilibrium phase at high temperature, and a hull
distance says nothing about how fast a decomposition would actually proceed.

### E-4. Practice 5
Three experiments on the entry set. Each one is a rebuild of `PhaseDiagram` from a filtered copy of
`pd_entries`, then a fresh `get_e_above_hull(target)`.

1. Remove the lowest-energy O2 entry, so the oxygen reference is a different calculation. Does the
   target's hull distance change? Explain what you find in terms of A-4.
2. Remove every entry whose hull distance is above 0.1 eV/atom. Does the target's hull distance
   change? Say which entries a convex hull is actually built from.
3. Rerun the C-1 query with `additional_criteria` left out, then rebuild. Compare the entry count, the
   number of entry ids ending in `-r2SCAN`, the number of stable phases, and the target's hull
   distance. Which of the two hulls should you report, and what has to accompany the number?

In [ ]:
## Write your code here.

## What you should be able to do now

- Tell `energy_per_atom`, `formation_energy_per_atom` and `energy_above_hull` apart, and say which of
  them can be compared across compositions.
- Rank the calculated structures at one composition and judge whether the ranking is resolved or
  inside the error of the method.
- Query a full chemical system, build the hull yourself, and check it against the database before
  trusting it.
- Turn a hull distance into a balanced decomposition reaction and verify the element balance.
- State what a hull distance does not tell you: nothing about temperature, nothing about kinetics, and
  nothing at all without the entry set and database version it was measured against.